# Lab 2 — The ROP Model That Can't Help You Drill

**Modules 4.1 and 4.2 — Drilling Optimization and Predictive Drilling Optimization**

A vendor offers a model that predicts rate of penetration from surface drilling parameters. The
pitch: feed it live rig data and it will tell you how to drill faster.

You are the drilling engineer evaluating it before it goes on the rig.

**Data:** Utah FORGE well 58-32, Milford, Utah. Real Pason EDR data — 7,311 depth samples from
85 ft to 7,536 ft, released by the US Department of Energy under CC BY 4.0.

*This is a geothermal well. The rig, the EDR, the channels and the drilling physics are the same as
an oil and gas well; the formation and the objective are not.*

**Run each cell with Shift+Enter.** You are not expected to write code. You are expected to decide
whether this model belongs on a rig.


> **Kernel:** use **conda_python3**. The cell below installs anything missing.


In [ ]:
import subprocess, sys

for module, package in [("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        __import__(module)
    except ImportError:
        print(f"installing {package} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package])

print("Ready.")


## Step 1 — The data, and what is wrong with it


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

df = pd.read_csv("data/forge_58-32_drilling.csv")

print(f"{len(df):,} depth samples, {df.DEPTH_FT.min():.0f} to {df.DEPTH_FT.max():.0f} ft")
df.head()


Before modelling anything, look at the ranges. Real EDR data arrives dirty, and no model will tell
you that — it will happily learn from nonsense.


In [ ]:
df.describe().loc[["min", "max", "mean"]].T.round(1)


> ### ⏸ Discuss (3 min)
>
> Three things in that table should bother a drilling engineer.
>
> - Look at **WH_PRESSURE_PSI**. Is that a physical measurement?
> - Look at the **maximum ROP_FTHR**. Is that a real rate of penetration?
> - **WOB_KLBS** and **RPM** both reach a minimum of 0. What is the rig doing at those moments?


In [ ]:
print("Dead channel — wellhead pressure:")
print(f"  range {df.WH_PRESSURE_PSI.min():.1f} to {df.WH_PRESSURE_PSI.max():.1f} psi, "
      f"mean {df.WH_PRESSURE_PSI.mean():.1f}")
print()
print("Implausible ROP:")
print(f"  {(df.ROP_FTHR > 500).sum()} samples above 500 ft/hr (max {df.ROP_FTHR.max():.0f})")
print()
print("Off-bottom samples — the bit is not cutting rock:")
print(f"  WOB = 0 : {(df.WOB_KLBS == 0).sum()} samples")
print(f"  RPM = 0 : {(df.RPM == 0).sum()} samples")


A negative wellhead pressure is a sensor offset, not a measurement. ROP near 3,000 ft/hr is a
recording artifact. And when weight on bit is zero the bit is off bottom — those rows describe
tripping, not drilling.

**We filter all three out.** A model trained on them would learn the rig's bookkeeping instead of
the rock.


In [ ]:
drilling = df[(df.WOB_KLBS > 0) & (df.RPM > 0) &
              (df.ROP_FTHR > 0) & (df.ROP_FTHR < 500)].sort_values("DEPTH_FT")

print(f"{len(df):,} raw samples  ->  {len(drilling):,} genuine on-bottom drilling samples")
print(f"discarded {len(df) - len(drilling):,} ({(1 - len(drilling)/len(df)) * 100:.0f}%)")


## Step 2 — Train the model the tempting way


The obvious approach: take all the drilling data, hold out a random 30%, train on the rest.


In [ ]:
FEATURES = ["WOB_KLBS", "RPM", "FLOW_IN_GPM", "PUMP_PSI", "TORQUE", "HOOKLOAD_KLBS"]

X_train, X_test, y_train, y_test = train_test_split(
    drilling[FEATURES], drilling.ROP_FTHR, test_size=0.3, random_state=42)

random_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
random_model.fit(X_train, y_train)
random_pred = random_model.predict(X_test)

print(f"R-squared:  {r2_score(y_test, random_pred):.3f}")
print(f"Mean error: {mean_absolute_error(y_test, random_pred):.1f} ft/hr")


> ### ⏸ Discuss (2 min)
>
> R-squared of 1.0 is perfect. **Would you put this on a rig?**
>
> Commit to an answer before running the next cell.


## Step 3 — Train it the way it would actually be used


A random split lets the model train on 5,000 ft and 5,100 ft while being tested on 5,050 ft. It has
already seen the rock on both sides of every answer.

**That is not the job.** On a rig the model is given the section you have already drilled and asked
about the section ahead of the bit. So split by depth: train on the upper 70% of the well, predict
the bottom 30%.


In [ ]:
split_at = int(len(drilling) * 0.7)
upper = drilling.iloc[:split_at]
lower = drilling.iloc[split_at:]

print(f"Train on:  {upper.DEPTH_FT.min():>6.0f} - {upper.DEPTH_FT.max():>6.0f} ft   ({len(upper):,} samples)")
print(f"Predict:   {lower.DEPTH_FT.min():>6.0f} - {lower.DEPTH_FT.max():>6.0f} ft   ({len(lower):,} samples)")

depth_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
depth_model.fit(upper[FEATURES], upper.ROP_FTHR)
depth_pred = depth_model.predict(lower[FEATURES])

print()
print(f"R-squared ahead of the bit: {r2_score(lower.ROP_FTHR, depth_pred):.3f}")


In [ ]:
print(f'Random split  (sees rock on both sides)  R2 = {r2_score(y_test, random_pred):>7.3f}')
print(f'Depth split   (predicts ahead of bit)    R2 = {r2_score(lower.ROP_FTHR, depth_pred):>7.3f}')


> ### ⏸ Discuss (4 min)
>
> A negative R-squared means the model's predictions are further from the truth than simply
> guessing the average would have been.
>
> - Nothing about the model changed between those two numbers. **Only how we tested it.** Which
>   number would a vendor put on a slide?
> - What would it take to make the second number honest — a better model, or more data?


### Why it collapses


In [ ]:
print(f"Average ROP, upper section:  {upper.ROP_FTHR.mean():5.1f} ft/hr")
print(f"Average ROP, lower section:  {lower.ROP_FTHR.mean():5.1f} ft/hr")
print()
print("The rock changed. The model never saw this formation.")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
ax.plot(lower.ROP_FTHR, lower.DEPTH_FT, lw=0.5, color="black", label="Actual ROP")
ax.plot(depth_pred, lower.DEPTH_FT, lw=0.5, color="crimson", alpha=0.8, label="Model prediction")
ax.axhline(upper.DEPTH_FT.max(), color="steelblue", ls="--", lw=1,
           label="End of training data")
ax.invert_yaxis()
ax.set_xlabel("ROP (ft/hr)")
ax.set_ylabel("Depth (ft)")
ax.set_title("Predicting ahead of the bit")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Step 4 — The question that actually matters


Set accuracy aside for a moment. The pitch was that this model helps you **drill faster**. So ask
what it is keying on — and whether those are things a driller can change.


In [ ]:
importance = (pd.Series(random_model.feature_importances_, index=FEATURES)
                .sort_values(ascending=False))

for name, value in importance.items():
    bar = "#" * int(value * 60)
    print(f"{name:16s} {value:5.3f}  {bar}")


> ### ⏸ Discuss (5 min) — this is the point of the lab
>
> A driller on tour can change **weight on bit**, **rotary speed**, and **flow rate**. Pump pressure
> and hookload are mostly consequences — of hole depth, hydraulics, and string weight.
>
> - Which variables dominate this model?
> - Where did **weight on bit** come out?
> - The model predicts ROP. **Does it tell the driller what to do differently?**
> - If you raised weight on bit on the rig, would this model's prediction follow?


In [ ]:
controllable = ["WOB_KLBS", "RPM", "FLOW_IN_GPM"]
consequence = ["PUMP_PSI", "TORQUE", "HOOKLOAD_KLBS"]

print(f"Levers a driller can pull:   {importance[controllable].sum() * 100:4.1f}% of the model")
print(f"Consequences of the hole:    {importance[consequence].sum() * 100:4.1f}% of the model")
print()
print(f"Weight on bit alone:         {importance['WOB_KLBS'] * 100:4.1f}%")


## Step 5 — Your decision

Agree as a group. 3 minutes.

1. **Would you put this model on a rig?** Yes, no, or yes with conditions.
2. **Which number would the vendor show you, and which would you ask for?**
3. A model can be accurate and still useless for decisions. **In one sentence, why?**
4. **What would you require** before trusting an ROP model ahead of the bit?

Be ready to answer in 60 seconds.


---

### Optional — if your group finishes early

Train on **only the levers a driller controls** and see how much predictive power was coming from
the consequence variables.


In [ ]:
levers_only = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
levers_only.fit(X_train[controllable], y_train)
levers_pred = levers_only.predict(X_test[controllable])

print(f"All six variables:        R2 = {r2_score(y_test, random_pred):.3f}")
print(f"Only driller's levers:    R2 = {r2_score(y_test, levers_pred):.3f}")


Look carefully. Feature importance said weight on bit was worth **2.3%** — yet dropping pump
pressure, torque and hookload entirely costs only about 0.14 of R-squared.

Both things are true, and the reason matters: **feature importance is not a measure of what causes
what.** When several inputs carry overlapping information, a tree model hands the credit to
whichever one splits most cleanly and starves the rest. Remove the greedy variable and the others
pick up the slack.

So "weight on bit scored 2.3%" does **not** mean weight on bit barely affects ROP. It means this
model did not need WOB to make its predictions, because pump pressure already told it where in the
hole it was.

**A model can rank your inputs confidently and still tell you nothing about which lever to pull.**
That is a question about causation, and it is answered by a drilling test — change one parameter,
hold the rest, measure the response — not by reading importances off a trained model.
